# Biogen data preprocessing

Biogen data are already log-transformed:
1. Standardise column names (SMILES → smiles, LOG HLM_CLint → value)
2. Canonical SMILES conversion
3. Duplicate check
4. Save as the smart_deduplicated version

In [1]:
import pandas as pd
import numpy as np
import os
from rdkit import Chem

input_dir = r'../data/in_vitro_data_deduplicated'
output_dir = r'../data/in_vitro_data_deduplicated'

input_file = 'biogen_CL_raw_deduplicated.csv'
output_file = 'biogen_CL_raw_smart_deduplicated.csv'

print("Biogen data preprocessing\n")

Biogen data preprocessing



## Step 1: Load and standardise column names

In [2]:
input_path = os.path.join(input_dir, input_file)
df = pd.read_csv(input_path)

print(f"Original file: {input_file}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

# Standardise column names
print("Standardizing column names:")

# SMILES → smiles
if 'SMILES' in df.columns:
    df.rename(columns={'SMILES': 'smiles'}, inplace=True)
    print("  ✓ SMILES → smiles")

# LOG HLM_CLint → value (already log-transformed)
if 'LOG HLM_CLint (mL/min/kg)' in df.columns:
    df.rename(columns={'LOG HLM_CLint (mL/min/kg)': 'value'}, inplace=True)
    print("  ✓ LOG HLM_CLint (mL/min/kg) → value")

print(f"\nNew columns: {list(df.columns)}")

Original file: biogen_CL_raw_deduplicated.csv
Rows: 3521
Columns: ['Internal ID', 'Vendor ID', 'SMILES', 'CollectionName', 'LOG HLM_CLint (mL/min/kg)', 'LOG MDR1-MDCK ER (B-A/A-B)', 'LOG SOLUBILITY PH 6.8 (ug/mL)', 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)', 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)', 'LOG RLM_CLint (mL/min/kg)']

Standardizing column names:
  ✓ SMILES → smiles
  ✓ LOG HLM_CLint (mL/min/kg) → value

New columns: ['Internal ID', 'Vendor ID', 'smiles', 'CollectionName', 'value', 'LOG MDR1-MDCK ER (B-A/A-B)', 'LOG SOLUBILITY PH 6.8 (ug/mL)', 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)', 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)', 'LOG RLM_CLint (mL/min/kg)']


## Step 2: Canonical SMILES conversion

In [3]:
def get_canonical_smiles(smiles_str):
    if pd.isna(smiles_str):
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return Chem.MolToSmiles(mol)
        else:
            return None
    except:
        return None

print("Converting to canonical SMILES...")
df['canonical_smiles'] = df['smiles'].apply(get_canonical_smiles)

# Remove invalid SMILES
invalid = df['canonical_smiles'].isna().sum()
df_valid = df[df['canonical_smiles'].notna()].copy()

print(f"  Valid SMILES: {len(df_valid)}/{len(df)} (removed {invalid})")

# Replace the original column with canonical SMILES
df_valid['smiles'] = df_valid['canonical_smiles']
df_valid = df_valid.drop('canonical_smiles', axis=1)

df = df_valid.copy()

Converting to canonical SMILES...
  Valid SMILES: 3521/3521 (removed 0)


## Step 3: Duplicate check

In [4]:
print("\nChecking for duplicates...")

# Whole-row duplicates
row_duplicates = df.duplicated(keep=False).sum()
print(f"  Full row duplicates: {row_duplicates}")

# SMILES duplicates
smiles_duplicates = df['smiles'].duplicated(keep=False).sum()
print(f"  SMILES duplicates: {smiles_duplicates}")

if smiles_duplicates > 0:
    print(f"\n  Merging {smiles_duplicates} duplicate SMILES...")
    
    # Group by canonical SMILES and take mean
    df = df.groupby('smiles').agg({'value': 'mean'}).reset_index()
    
    # Copy other columns (assuming they're the same for duplicates)
    # For now, just keep smiles and value
    
    print(f"  After merging: {len(df)} rows")
else:
    print("  ✓ No duplicates to merge")


Checking for duplicates...
  Full row duplicates: 0
  SMILES duplicates: 0
  ✓ No duplicates to merge


## Step 4: Save

In [5]:
output_path = os.path.join(output_dir, output_file)
df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {list(df.columns)}")

# Data validation
print(f"\nData preview:")
print(f"  Value stats:")
print(f"    Mean: {df['value'].mean():.4f}")
print(f"    Std: {df['value'].std():.4f}")
print(f"    Min: {df['value'].min():.4f}")
print(f"    Max: {df['value'].max():.4f}")
print(f"\n  First 3 rows:")
print(df.head(3))


✓ Saved: ..\data\in_vitro_data_deduplicated\biogen_CL_raw_smart_deduplicated.csv
  Rows: 3521
  Columns: ['Internal ID', 'Vendor ID', 'smiles', 'CollectionName', 'value', 'LOG MDR1-MDCK ER (B-A/A-B)', 'LOG SOLUBILITY PH 6.8 (ug/mL)', 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)', 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)', 'LOG RLM_CLint (mL/min/kg)']

Data preview:
  Value stats:
    Mean: 1.3200
    Std: 0.6240
    Min: 0.6757
    Max: 3.3727

  First 3 rows:
  Internal ID  Vendor ID                                             smiles  \
0        Mol1  317714313  CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...   
1        Mol2  324056965  CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1   
2        Mol3  304005766                        CN(c1ncc(F)cn1)[C@H]1CCCNC1   

  CollectionName     value  LOG MDR1-MDCK ER (B-A/A-B)  \
0     emolecules  0.675687                    1.493167   
1     emolecules  0.675687                    1.040780   
2     emolecules  0.675687         